# LogXY ForestPersons Published Baseline Smoke Test

This notebook prepares a restart-safe Google Colab Free/T4 workflow for COCO-pretrained MMDetection DETR R50 and Deformable Faster R-CNN R50-FPN/DCN smoke inference on 30 ForestPersons val600 images.

It does **not** fine-tune models and does **not** claim published ForestPersons baseline reproduction. Outputs are saved in Google Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Set Project Paths
Upload or sync the LogXY project directory to `MyDrive/LogXY` before running. The directory must contain `data/images`, the val600 one-class COCO export, scripts, and this CUDA experiment folder.


In [ ]:
import json, os, pathlib
LOGXY_ROOT = '/content/drive/MyDrive/LogXY'
os.environ['LOGXY_ROOT'] = LOGXY_ROOT
os.environ['MMDET_ROOT'] = '/content/third_party/mmdetection'
os.environ['LOGXY_MAMBA_ENV'] = 'logxy-mmdet2'
root = pathlib.Path(LOGXY_ROOT)
required = [
    'outputs/baseline_feasibility/forestpersons_val600_oneclass_coco.json',
    'outputs/baseline_feasibility/forestpersons_val600_smoke30_manifest.json',
    'data/annotations/val.json',
    'src/forestpersons/evaluation.py',
    'scripts/mmdet_infer_forestpersons.py',
    'scripts/evaluate_external_coco_results.py',
    'experiments/published_baselines_cuda/colab/setup_mmdet2_colab.sh',
]
missing = [path for path in required if not (root / path).exists()]
assert not missing, 'Missing required files: ' + repr(missing)
coco = json.loads((root / 'outputs/baseline_feasibility/forestpersons_val600_oneclass_coco.json').read_text())
smoke = json.loads((root / 'outputs/baseline_feasibility/forestpersons_val600_smoke30_manifest.json').read_text())
assert len(coco['images']) >= 30, len(coco['images'])
assert len(smoke['images']) == 30, len(smoke['images'])
missing_images = [img['file_name'] for img in coco['images'][:30] if not (root / 'data/images' / img['file_name']).exists()]
assert not missing_images, 'Missing smoke images: ' + repr(missing_images[:5])
print('Project root ready:', LOGXY_ROOT)
print('Smoke images verified:', len(coco['images'][:30]))


In [ ]:
!nvidia-smi


## Install Isolated MMDetection 2.x Environment
This creates a Python 3.10 micromamba environment with PyTorch 1.13.1 + CUDA 11.7, MMCV 1.7.1, and MMDetection v2.28.2. It may take several minutes. Re-running after a runtime restart reuses existing Drive data and recreates only the ephemeral VM environment.


In [ ]:
!bash $LOGXY_ROOT/experiments/published_baselines_cuda/colab/setup_mmdet2_colab.sh


In [ ]:
!export PATH="$HOME/.local/bin:$PATH"; export MAMBA_ROOT_PREFIX="${MAMBA_ROOT_PREFIX:-$HOME/micromamba}"; cd $LOGXY_ROOT && micromamba run -n logxy-mmdet2 python scripts/verify_mmdet_cuda_env.py


## Download COCO-Pretrained Checkpoints
The checkpoints persist under `MyDrive/LogXY/experiments/published_baselines_cuda/checkpoints`. These are COCO-pretrained weights, not ForestPersons-trained weights.


In [ ]:
!bash $LOGXY_ROOT/experiments/published_baselines_cuda/colab/download_checkpoints_colab.sh


## Run 30-Image Smoke Test
This runs DETR R50 and Deformable Faster R-CNN R50-FPN/DCN on the first 30 images in the val600 COCO export, writes COCO result JSON, and evaluates through the existing LogXY external-results evaluator.


In [ ]:
!bash $LOGXY_ROOT/experiments/published_baselines_cuda/colab/run_smoke30_colab.sh


In [ ]:
!find $LOGXY_ROOT/experiments/published_baselines_cuda/results/colab_smoke30 -maxdepth 2 -type f | sort
